# ComfyUI + WAN 2.2 Smooth Workflow v5.0 — Colab Pro (A100)

**DigitalPastel'in Smooth Workflow v5.0** (`WAN_2_2_Smooth_Workflow_v5_0.json`) ile image-to-video. ComfyUI arka planda **API** olarak çalışır (localhost, Cloudflare yok); `input/` klasöründeki görselleri otomatik işler.

**Drive yapısı** (`MyDrive/ImageToVideo/`):
```
ImageToVideo/
├── imageToVideo.json   ← API-format workflow
├── input/              ← işlenecek TÜM görseller (prompt dosya numarasına göre ACTION_PROMPTS'tan seçilir)
└── output/             ← üretilen videolar (isim input ile eşleşir; resume: var olanı atlar)
```

**Kullanım sırası:**
1. Runtime → Change runtime type → **A100 GPU** seç (Pro/Pro+ gerekli)
2. **CONFIG** hücresinde `CIVITAI_API_KEY`, `COOKIE_VALUE` ve `ACTION_PROMPTS` (dosya numarasına göre prompt) ayarlarını gir
3. Tüm hücreleri sırayla çalıştır (Runtime → Run all)
4. Son hücreler input görsellerini API üzerinden işleyip çıktıları Drive'daki `output/`'a kaydeder

Sıra:
1. **CONFIG** — API key + cookie + ACTION_PROMPTS + render ayarları
2. **Google Drive Mount + Input Tarama**
3. **ComfyUI + Tüm Custom Node'lar**
4. **Modeller** — HuggingFace (ücretsiz) + SmoothMix checkpoint'leri (Civitai)
5. **ComfyUI'yi Başlat** (arka planda, localhost)
6. **API Modülleri**
7. **Çalıştır** (batch + resume)
8. **Önizleme**

> Her indirme/kurulum adımı problemde **anında hata fırlatır** (fail-loud) — bozuk model sessizce diske kalmaz.

## 1) CONFIG — Tek hücre, hepsini buradan kontrol et

Aşağıdaki değişkenleri düzenle, diğer hücrelere dokunma. Tamamlandığında **Runtime → Run all**.

In [ ]:
# === Civitai — model download credentials ===

# API key (civitai.com → Account → API Keys)
CIVITAI_API_KEY = "c2f8440d15d825229098e9a5a75c5dfa"

# Cookie: civitai.red login → F12 → Application → Cookies → __Secure-civitai-token
COOKIE_VALUE = "eyJhbGciOiJkaXIiLCJlbmMiOiJBMjU2R0NNIn0..SFCFnF5UuVIp7441.KDMRVEUBifCwNBvC4rHpNDZ4r25kDv_n5BsEjWGg4-OI7m07-3nZppI8jd8eNz4rQw4iZnQ7kYOG2_8mb-VgolfMSAQF9kvWAqYdGpvf2jG_UJGMyCbqqnO8mFMkz7Ptu_sHp3CZ7ffX10Lmm4KrqXSrKFGw9hQUKN-h5lJdmkSIqqc3idArF6ZfyI3ZwVhHfROKz-IHjsyf08iScmCa2mOTwFB75T_BbH1U7o_l3dssJP7VCRzirO8YXtYFP3FTSaA9vkVG-8XqWgO6A3K5w5KgS2c-6K3uSr5ozVyMv7DHnXa1rNNvsF9fLZdtP0N7DC39_1gIeGjnlXX3XP9XZEJssK9U1h89XPBbAK9xfQckkXsIUICaiIOXLXOK0YEDjTcVQDz7IaBGE_tKGo-bwwEwtddVIsP82f0scZFnNiFryUQhPPe6sWlAyEx7cvZSaTDur-wUEvEoBzUYbJ5R0SrMKkfGpv3EeYPx-L0tZSCFQcbdvxJRnkndm20HV4GznM2Jl0muwF2WMy-0K5ozNNI8p8I10vE4-927XiGoCGP6YbayTm3cJO7j_UUgTEg85qkGFZmtmTzW5D11pHAYKO5h9szesu2pSwsJDOqEeq_GWt2KpvbMWyVjiCtk9rmGERL053NdqKqdTP5nnRtn4m5V1r3n-yWh2y_jowAcHHI2moBnUOqdhhYumeH-_ggoNWtcu4mUD1ucBLQwkHcScqc1j8Ca-HsAoeExiTq_VrbYnAKQ0_NlxI3YDplsASRK5ZMea37G482d8Gxq3e2DQjBoQ58__TM4vaTf2h03xcV0jX67AVDjY9lRhAKINfggKbWm78zkpBKjrj4ZvcSv-WEFrbt5szoBJ31WH4H6S9X4iRkTjyXdZvY8Wlk0-X-OX_SCr_V1GW1iXzOBjCb4a1x9xXmiU0hOZ4eVqhgjdbFdtu_m2veBAswcdMWepDTHsGp3tAVoGKCy_CNOkwWlggyliiW7mq52z39aug69HFZ6fxdU_eNNEZUic2q9mefSS98MmTefnJrpvaVoZv2KR6UKD9tVkvex98jgwJCtN3-3pXRoVtH2LM-JhX-dJfgj7qvf8QLVWiwHgIUIVYgqWHKi8yPd5tvzJnNTNeqpD29bXkx-qbmQyQ.nWm_Jj6PxOl-XIEyGEwMaA"

# === Drive — workflow + input + output ===
DRIVE_BASE        = "/content/drive/MyDrive/ImageToVideo"
WORKFLOW_FILENAME = "imageToVideo.json"   # under DRIVE_BASE

# === Render — action prompts ===
# Order = file number: ACTION_PROMPTS[0] -> 01.png, ACTION_PROMPTS[1] -> 02.png, ...
# Matching logic (image_number/prompt_for_image) lives in section 2 — values only here.
ACTION_PROMPTS = [
    "",   # 01.png
    "",   # 02.png
    "",   # 03.png
    # add / remove lines as needed
]

# Images whose number exceeds the list / have no number render with this prompt.
DEFAULT_PROMPT = ""

# === Render — API settings ===
NUM_RENDERS = 1            # renders per image — 3 photos × NUM_RENDERS=2 = 6 videos
SEED_START  = None         # None = random | int = fixed (each render +1)

# === LoRA strengths ===
LORA_STRENGTHS = {
    "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors": 1.0,
    "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors":  1.0,
    "WAN_General_NSFW_HIGH.safetensors": 1.0,
    "WAN_General_NSFW_LOW.safetensors":  1.0,
    "Slop_Bounce_I2V_High.safetensors":  1.0,
    "Slop_Bounce_I2V_Low.safetensors":   1.0,
}

# === Derived paths + validation + env ===
import os

os.environ['CIVITAI_API_KEY'] = CIVITAI_API_KEY
os.environ['COOKIE_VALUE']    = COOKIE_VALUE

WORKFLOW_PATH = f"{DRIVE_BASE}/{WORKFLOW_FILENAME}"
INPUT_DIR     = f"{DRIVE_BASE}/input"
OUTPUT_DIR    = f"{DRIVE_BASE}/output"
COMFYUI_URL   = "http://127.0.0.1:8188"

assert 'PASTE' not in CIVITAI_API_KEY, "CIVITAI_API_KEY girilmemiş"
assert 'PASTE' not in COOKIE_VALUE, "COOKIE_VALUE girilmemiş"
assert len(COOKIE_VALUE) > 500, "Cookie çok kısa"

print(f"✓ API key: {len(CIVITAI_API_KEY)} char  |  Cookie: {len(COOKIE_VALUE)} char")
print(f"✓ Render: {NUM_RENDERS} render/görsel, seed={SEED_START or 'random'}")
print(f"✓ Drive base: {DRIVE_BASE}")
!nvidia-smi

## 2) Google Drive Mount + Input Tarama

Drive mount edilir, `input/` taranır ve her görsel dosya numarasına göre prompt'una eşlenir. (Drive yapısı en üstteki başlıkta.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, glob
from pathlib import Path

# === Prompt matching (file number -> ACTION_PROMPTS) ===
def image_number(filename):
    """First number in the filename (05.png -> 5, scene_07.png -> 7). None if absent."""
    stem = os.path.splitext(os.path.basename(filename))[0]
    m = re.search(r"\d+", stem)
    return int(m.group()) if m else None

def prompt_for_image(path):
    """Prompt by file number: N -> ACTION_PROMPTS[N-1].
    No number / N<1 / out of range -> DEFAULT_PROMPT."""
    n = image_number(path)
    if n is None or n < 1 or n - 1 >= len(ACTION_PROMPTS):
        return DEFAULT_PROMPT
    return ACTION_PROMPTS[n - 1]

# === Drive setup + input scan ===
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert os.path.exists(WORKFLOW_PATH), f"Workflow yok: {WORKFLOW_PATH}"

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
INPUT_IMAGES = sorted([
    p for p in glob.glob(f"{INPUT_DIR}/*")
    if Path(p).suffix.lower() in IMAGE_EXTS
])
assert INPUT_IMAGES, f"Input klasöründe görsel yok: {INPUT_DIR}"

# === Image <-> prompt preview ===
print(f"✓ Workflow: {WORKFLOW_PATH}")
print(f"✓ Output:   {OUTPUT_DIR}")
print(f"✓ {len(INPUT_IMAGES)} görsel, {len(ACTION_PROMPTS)} action prompt\n")
print("📝 Görsel ↔ prompt (dosya numarasına göre):")
for p in INPUT_IMAGES:
    n  = image_number(p)
    pr = prompt_for_image(p)
    tag = f"#{n}" if (n and n - 1 < len(ACTION_PROMPTS)) else "DEF"
    show = (pr[:70] + "…") if len(pr) > 70 else (pr or "(boş)")
    print(f"   {Path(p).name:18s} [{tag:>4s}] -> {show}")
print(f"\nToplam render: {len(INPUT_IMAGES)} × {NUM_RENDERS} = {len(INPUT_IMAGES) * NUM_RENDERS} video")

In [ ]:
# === Shared helpers — log + fail-loud run + safetensors validation ===
# Used by section 3 (custom nodes) and section 4 (model download); defined once (DRY).
import os, time, struct, subprocess

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError (fail-loud).
    cmd: str = shell, list = argv."""
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def is_valid_safetensors(path):
    """Is the file a real safetensors? -> (ok: bool, msg: str).
    Empty / HTML / JSON error pages are caught as corrupt — root cause of the UNETLoader JSONDecodeError."""
    if not os.path.exists(path):
        return False, "yok"
    sz = os.path.getsize(path)
    if sz < 1_000_000:
        return False, f"çok küçük ({human(sz)})"
    with open(path, "rb") as f:
        head = f.read(8)
    if head.startswith(b"<") or head.startswith(b'{"'):
        return False, "HTML/JSON hata sayfası"
    try:
        jl = struct.unpack("<Q", head)[0]
        if 100 < jl < 200_000_000:
            return True, f"valid ({human(sz)})"
    except Exception:
        pass
    return False, "header bozuk"

print("✓ Ortak yardımcılar hazır (log, run, human, is_valid_safetensors)")

## 3) ComfyUI + Tüm Custom Node'lar

Workflow v5.0'ın istediği custom node'lar tek seferde kurulur — hangi node'un ne sağladığı kod içinde inline yorumda. Biri başarısız olursa hücre durur (fail-loud).

In [ ]:
%cd /content

# === System deps + ComfyUI ===
!apt-get install -y ffmpeg aria2 > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!pip install -q opencv-python imageio imageio-ffmpeg sageattention

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
%cd /content/ComfyUI/custom_nodes

# (folder, repo) — trailing comment = what the node provides
CUSTOM_NODES = [
    ("ComfyUI-Manager",                 "https://github.com/ltdrdata/ComfyUI-Manager.git"),            # detect missing nodes
    ("rgthree-comfy",                   "https://github.com/rgthree/rgthree-comfy.git"),               # Power Lora Loader, Seed, Fast Groups Bypasser
    ("comfy_mtb",                       "https://github.com/melMass/comfy_mtb.git"),                   # Note Plus, Pick From Batch
    ("ComfyUI-VideoHelperSuite",        "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),# VHS_LoadVideo, VHS_VideoCombine
    ("ComfyUI-MMAudio",                 "https://github.com/kijai/ComfyUI-MMAudio.git"),               # audio generation
    ("ComfyUI-WanVideoWrapper",         "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),       # Wan video nodes
    ("ComfyUI-GGUF",                    "https://github.com/city96/ComfyUI-GGUF.git"),                 # UnetLoaderGGUF (quantized loader)
    ("ComfyUI-KJNodes",                 "https://github.com/kijai/ComfyUI-KJNodes.git"),               # ImageResizeKJv2, ColorMatch
    ("ComfyMath",                       "https://github.com/evanspearman/ComfyMath.git"),             # math expressions
    ("ComfyUI-Frame-Interpolation",     "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git"),# RIFE / frame interpolation
    ("ComfyUI-VFI",                     "https://github.com/GACLove/ComfyUI-VFI.git"),                 # frame interpolation
    ("ComfyUI_Comfyroll_CustomNodes",   "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git"),# CR Float To Integer
    ("ComfyUI-Easy-Use",                "https://github.com/yolain/ComfyUI-Easy-Use.git"),            # cleanGpuUsed
    ("ComfyUI-mxToolkit",               "https://github.com/Smirnov75/ComfyUI-mxToolkit.git"),        # slider widgets
    ("ComfyUI-NAG",                     "https://github.com/scottmudge/ComfyUI-NAG.git"),             # normalized attention guidance
    ("comfyui-adaptiveprompts",         "https://github.com/Alectriciti/comfyui-adaptiveprompts.git"),# dynamic prompts
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    run(["git", "clone", "--depth", "1", url, name], f"clone {name}", timeout=120)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=300)  # install node deps

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## 4) Modeller — HuggingFace (ücretsiz) + SmoothMix (Civitai)

Tüm model dosyaları bu hücrede iner. **Herhangi biri bozuk/eksik inerse hücre `RuntimeError` ile durur** (sessiz devam yok — eski `UNETLoader → JSONDecodeError` bunun yüzünden oluşuyordu).

**HuggingFace (~40GB, `aria2c`):**
- Wan 2.2 base diffusion modelleri (high + low noise) — fallback
- Lightx2v 4-step LoRA'lar
- Wan 2.1 VAE → workflow `Wan2_1_VAE_fp32.safetensors` ismini bekler, o isimle iner
- UMT5-XXL text encoder + CLIP Vision H
- MMAudio modelleri (4 dosya, ses sentezi)

**SmoothMix checkpoint'leri (Civitai, `curl` — workflow bunları kullanır):**
- I2V v2.0 High → `2513182`  •  I2V v2.0 Low → `2513186` (workflow `SmoothMix_I2V_v2_*.safetensors` ister, o isimle iner)
- NSFW LoRA'lar: WAN General NSFW HIGH/LOW → `loras/` (Power LoRA Loader node'undan seçilir)

Kullanılmayan version ID'ler (T2V v3.0, Slop Bounce, DR34ML4Y) kodda yorum referansı olarak durur, indirilmez.

In [ ]:
import os, glob

# === Target folders ===
COMFY = "/content/ComfyUI"
DIFF  = f"{COMFY}/models/diffusion_models"
LORA  = f"{COMFY}/models/loras"
for d in ["diffusion_models", "loras", "text_encoders", "vae", "clip_vision", "mmaudio"]:
    os.makedirs(f"{COMFY}/models/{d}", exist_ok=True)

# === Single download function — shared flow for HF (aria2c) and Civitai (curl) (DRY) ===
def fetch(url, target_dir, filename, label, *, parallel, headers=None):
    """Download + validate a model. Skip if already valid; raise RuntimeError if the result is invalid (fail-loud).
    parallel=True -> aria2c (fast for large HF files), False -> curl (Civitai, surfaces HTTP status)."""
    target = os.path.join(target_dir, filename)
    ok, msg = is_valid_safetensors(target)
    if ok:
        log(f"{label}: zaten var ({msg})")
        return
    if os.path.exists(target):          # drop corrupt/partial leftover, then re-download
        os.remove(target)

    if parallel:
        cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M",
               "--console-log-level=warn", "--auto-file-renaming=false",
               "--allow-overwrite=true", "-d", target_dir, "-o", filename]
        if headers:
            cmd += ["--header", headers]
        cmd.append(url)
        run(cmd, label)
    else:
        # curl writes %{http_code} to stdout so a Civitai auth failure (401/403/NotEntitled)
        # reports the real status + response snippet instead of a blind 'exit 22'.
        cmd = ["curl", "-sL", "--max-time", "1800", "-w", "%{http_code}", "-o", target]
        if headers:
            cmd += ["-H", headers]
        cmd.append(url)
        code = (run(cmd, label) or "").strip()[-3:]
        if not code.startswith("2"):
            snippet = ""
            if os.path.exists(target):
                with open(target, "rb") as f:
                    snippet = f.read(200).decode("utf-8", "replace").replace("\n", " ").strip()
            raise RuntimeError(f"{label}: Civitai HTTP {code} — {snippet}")

    ok, msg = is_valid_safetensors(target)
    if not ok:
        raise RuntimeError(f"{label}: indirme bozuk ({msg}) — {url.split('?')[0]}")
    log(f"{label}: indirildi ({msg})", "OK")

# Auth = __Secure-civitai-token session cookie ONLY. The ?token= API key makes civitai.com
# authenticate as that key's account -> creator-login-gated assets reply 401. The browser
# session cookie (logged-in user) downloads them fine. Cookie expires -> refresh from civitai.red.
def civitai_url(version_id):
    return f"https://civitai.com/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civitai-token={COOKIE_VALUE}"

# === HuggingFace models (aria2c) ===
WAN22 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
WAN21 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"
KIJAI = "https://huggingface.co/Kijai/MMAudio_safetensors/resolve/main"

HF_MODELS = [
    # (url, target_dir, filename, label)
    (f"{WAN22}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", DIFF, "wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors", "Wan2.2 HIGH"),
    (f"{WAN22}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors",  DIFF, "wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors",  "Wan2.2 LOW"),
    (f"{WAN22}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors",   LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "Lightx2v HIGH"),
    (f"{WAN22}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",    LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",  "Lightx2v LOW"),
    # VAE: workflow expects 'Wan2_1_VAE_fp32.safetensors', so download the base under that name
    (f"{WAN21}/vae/wan_2.1_vae.safetensors",                          f"{COMFY}/models/vae",          "Wan2_1_VAE_fp32.safetensors",             "Wan2.1 VAE"),
    (f"{WAN21}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors", f"{COMFY}/models/text_encoders","umt5_xxl_fp8_e4m3fn_scaled.safetensors",   "UMT5-XXL"),
    (f"{WAN21}/clip_vision/clip_vision_h.safetensors",               f"{COMFY}/models/clip_vision",  "clip_vision_h.safetensors",               "CLIP Vision H"),
    (f"{KIJAI}/mmaudio_large_44k_v2_fp16.safetensors",               f"{COMFY}/models/mmaudio",      "mmaudio_large_44k_v2_fp16.safetensors",   "MMAudio large"),
    (f"{KIJAI}/mmaudio_vae_44k_fp16.safetensors",                    f"{COMFY}/models/mmaudio",      "mmaudio_vae_44k_fp16.safetensors",        "MMAudio VAE"),
    (f"{KIJAI}/mmaudio_synchformer_fp16.safetensors",                f"{COMFY}/models/mmaudio",      "mmaudio_synchformer_fp16.safetensors",    "MMAudio synchformer"),
    (f"{KIJAI}/apple_DFN5B-CLIP-ViT-H-14-384_fp16.safetensors",      f"{COMFY}/models/mmaudio",      "apple_DFN5B-CLIP-ViT-H-14-384_fp16.safetensors", "MMAudio CLIP"),
]
for url, d, fn, label in HF_MODELS:
    fetch(url, d, fn, label, parallel=True)

# === SmoothMix checkpoints + NSFW LoRA (Civitai, curl + login cookie) ===
# Active (used by the workflow). Unused reference IDs: T2V v3 High 2768924 / Low 2768944;
# Slop Bounce High 2209354 / Low 2209344; DR34ML4Y High 2553151 / Low 2553271.
CIVITAI_MODELS = [
    # (version_id, target_dir, filename, label)
    (2513182, DIFF, "SmoothMix_I2V_v2_High.safetensors", "SmoothMix I2V v2 HIGH"),
    (2513186, DIFF, "SmoothMix_I2V_v2_Low.safetensors",  "SmoothMix I2V v2 LOW"),
    (2073605, LORA, "WAN_General_NSFW_HIGH.safetensors", "WAN General NSFW HIGH"),
    (2083303, LORA, "WAN_General_NSFW_LOW.safetensors",  "WAN General NSFW LOW"),
]
for vid, d, fn, label in CIVITAI_MODELS:
    fetch(civitai_url(vid), d, fn, label, parallel=False, headers=cookie_header())

# === Summary (reaching here means everything downloaded + validated) ===
print("\n📂 diffusion_models/")
for f in sorted(glob.glob(f"{DIFF}/*.safetensors")):
    print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
print("📂 loras/")
for f in sorted(glob.glob(f"{LORA}/*.safetensors")):
    print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
log("Tüm modeller indirildi ve doğrulandı", "OK")

## 5) ComfyUI'yi Başlat (Arka Planda)

ComfyUI subprocess olarak arka planda başlar; API client localhost'a bağlanır. **90s içinde hazır olmazsa hücre `RuntimeError` ile durur** (sonraki hücreler ölü sunucuya çalışmasın).

In [ ]:
import subprocess, time, os, urllib.request

# === Kill old instance ===
os.system("pkill -f 'python main.py' 2>/dev/null")
time.sleep(2)

# === Start in background (logs to file) ===
log_path = "/content/comfyui.log"
log_file = open(log_path, "w")
p = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188"],
    cwd="/content/ComfyUI",
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"▶ ComfyUI başlatıldı (PID {p.pid}), log: {log_path}")
print("⏳ Hazır olması bekleniyor...")

# === Ready? max 90s — otherwise fail-loud ===
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print(f"✅ ComfyUI hazır ({(i+1)*2}s)")
        break
    except Exception:
        pass
else:
    with open(log_path) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("ComfyUI 90s içinde başlamadı — yukarıdaki log'a bak")

## 6) API Modülleri

Her fonksiyon tek sorumluluk (SRP). Bir kere çalıştır, sonra dokunmasan yeter.

In [ ]:
import json, copy, time, uuid, random, requests, os, glob
from pathlib import Path

# === Template I/O ===
def load_workflow(path):
    with open(path) as f: return json.load(f)

def clone(workflow):
    return copy.deepcopy(workflow)

# === Patchers — each injects one workflow field (SRP) ===
POSITIVE_PROMPT_NODE = "333:291"
START_IMAGE_NODE     = "338"
END_IMAGE_NODE       = "342"
SEED_NODE            = "327"
LORA_LOADER_NODES    = ["324", "325"]

def set_prompt(workflow, prompt):
    workflow[POSITIVE_PROMPT_NODE]["inputs"]["prompt"] = prompt

def set_images(workflow, start_filename, end_filename):
    workflow[START_IMAGE_NODE]["inputs"]["image"] = start_filename
    workflow[END_IMAGE_NODE]["inputs"]["image"]   = end_filename

def set_seed(workflow, seed):
    workflow[SEED_NODE]["inputs"]["seed"] = seed

def set_lora_strengths(workflow, strength_map):
    for node_id in LORA_LOADER_NODES:
        node = workflow.get(node_id)
        if not node: continue
        for v in node["inputs"].values():
            if isinstance(v, dict) and "lora" in v:
                if v["lora"] in strength_map:
                    v["strength"] = strength_map[v["lora"]]

def apply_config(workflow, start_filename, end_filename, seed, prompt):
    set_prompt(workflow, prompt)
    set_images(workflow, start_filename, end_filename)
    set_seed(workflow, seed)
    set_lora_strengths(workflow, LORA_STRENGTHS)
    return workflow

# === ComfyUI HTTP client ===
class ComfyClient:
    def __init__(self, base_url):
        self.base = base_url.rstrip("/")
        self.client_id = str(uuid.uuid4())

    def upload_image(self, local_path):
        name = Path(local_path).name
        with open(local_path, "rb") as f:
            r = requests.post(
                f"{self.base}/upload/image",
                files={"image": (name, f)},
                data={"overwrite": "true"},
                timeout=60,
            )
        r.raise_for_status()
        return r.json()["name"]

    def submit(self, workflow):
        r = requests.post(
            f"{self.base}/prompt",
            json={"prompt": workflow, "client_id": self.client_id},
            timeout=30,
        )
        r.raise_for_status()
        data = r.json()
        if data.get("node_errors"):
            raise RuntimeError(f"Workflow hatası: {data['node_errors']}")
        return data["prompt_id"]

    def wait(self, prompt_id, poll_interval=2.0):
        last = ""
        while True:
            history = requests.get(f"{self.base}/history/{prompt_id}", timeout=10).json()
            if prompt_id in history:
                entry = history[prompt_id]
                status = entry.get("status", {})
                if status.get("status_str") == "error":
                    raise RuntimeError(f"ComfyUI error: {status.get('messages')}")
                return entry
            q = requests.get(f"{self.base}/queue", timeout=10).json()
            msg = f"⏳ running:{len(q.get('queue_running',[]))} pending:{len(q.get('queue_pending',[]))}"
            if msg != last:
                print(msg); last = msg
            time.sleep(poll_interval)

    def download_video(self, history_entry, save_path):
        for node_output in history_entry["outputs"].values():
            for key in ("gifs", "videos", "images"):
                for item in node_output.get(key, []):
                    if item.get("filename", "").lower().endswith((".mp4", ".webm", ".mov")):
                        params = {
                            "filename":  item["filename"],
                            "subfolder": item.get("subfolder", ""),
                            "type":      item.get("type", "output"),
                        }
                        r = requests.get(f"{self.base}/view", params=params, timeout=120)
                        r.raise_for_status()
                        with open(save_path, "wb") as f:
                            f.write(r.content)
                        return save_path
        raise RuntimeError("Video çıktısı bulunamadı")

# === Orchestrator — batch + resume + timing ===
def resolve_seed(index):
    if SEED_START is None:
        return random.randint(0, 2**31 - 1)
    return SEED_START + index

def output_path_for(input_image_path, seed):
    """Output path matching the input name.
    NUM_RENDERS=1 -> photo.mp4 | NUM_RENDERS>1 -> photo_seedX.mp4
    """
    stem = Path(input_image_path).stem
    if NUM_RENDERS == 1:
        return f"{OUTPUT_DIR}/{stem}.mp4"
    return f"{OUTPUT_DIR}/{stem}_seed{seed}.mp4"

def count_existing_renders(input_image_path):
    """How many renders already exist on disk for this image (resume)."""
    stem = Path(input_image_path).stem
    if NUM_RENDERS == 1:
        return 1 if os.path.exists(f"{OUTPUT_DIR}/{stem}.mp4") else 0
    return len(glob.glob(f"{OUTPUT_DIR}/{stem}_seed*.mp4"))

def render_one(client, template, image_remote_name, image_local_path, render_index, prompt):
    t0 = time.time()
    workflow = clone(template)
    seed = resolve_seed(render_index)
    apply_config(workflow, image_remote_name, image_remote_name, seed, prompt)
    save_path = output_path_for(image_local_path, seed)
    print(f"   → Render {render_index+1}/{NUM_RENDERS} (seed={seed})")
    pid = client.submit(workflow)
    history = client.wait(pid)
    client.download_video(history, save_path)
    elapsed = time.time() - t0
    print(f"   ✓ ({elapsed:.1f}s) → {Path(save_path).name}")
    return save_path, elapsed

def render_image(client, template, image_local_path, image_index, total_images):
    """Render (NUM_RENDERS - existing) clips for one image."""
    name = Path(image_local_path).name
    prompt = prompt_for_image(image_local_path)
    n = image_number(image_local_path)
    tag = f"#{n}" if (n and n - 1 < len(ACTION_PROMPTS)) else "DEFAULT"
    existing = count_existing_renders(image_local_path)
    remaining = max(0, NUM_RENDERS - existing)

    print(f"\n📸 Görsel {image_index+1}/{total_images}: {name}  (prompt: {tag})")

    if remaining == 0:
        print(f"   ⏭️  Tüm renderlar zaten var ({existing}/{NUM_RENDERS}), atlanıyor")
        return []
    if existing > 0:
        print(f"   📌 {existing}/{NUM_RENDERS} render zaten var, {remaining} yenisi yapılacak")

    image_remote = client.upload_image(image_local_path)
    results = []
    for i in range(remaining):
        # advance the index by the existing count to keep ordering correct in deterministic seed mode
        p, e = render_one(client, template, image_remote, image_local_path, existing + i, prompt)
        results.append((p, e))
    return results

def run():
    client = ComfyClient(COMFYUI_URL)
    template = load_workflow(WORKFLOW_PATH)
    total_t0 = time.time()
    all_results = []

    for idx, img_path in enumerate(INPUT_IMAGES):
        results = render_image(client, template, img_path, idx, len(INPUT_IMAGES))
        all_results.extend(results)

    total = time.time() - total_t0
    paths = [r[0] for r in all_results]
    elapseds = [r[1] for r in all_results]
    target_total = len(INPUT_IMAGES) * NUM_RENDERS

    print(f"\n{'='*60}")
    print(f"🎉 Bu run: {len(paths)} yeni video")
    print(f"   Hedef: {len(INPUT_IMAGES)} görsel × {NUM_RENDERS} render = {target_total} toplam")
    if elapseds:
        avg = sum(elapseds) / len(elapseds)
        print(f"   Süre: {total:.1f}s (ortalama {avg:.1f}s/video)")
    else:
        print(f"   Süre: {total:.1f}s (her şey zaten yapılmıştı)")
    return paths

print("✓ Modüller yüklendi (batch + resume mode)")

## 7) Çalıştır

In [ ]:
video_paths = run()

## 8) Önizleme — Inline

In [ ]:
from IPython.display import Video, display, Markdown
for p in video_paths:
    display(Markdown(f'### `{p}`'))
    display(Video(p, embed=True))